### What is Query Decomposition?
Query decomposition is the process of taking a complex, multi-part question and breaking it into simpler, atomic sub-questions that can each be retrieved and answered individually.

#### Why Use Query Decomposition?

- Complex queries often involve multiple concepts

- LLMs or retrievers may miss parts of the original question

- It enables multi-hop reasoning (answering in steps)

- Allows parallelism (especially in multi-agent frameworks)

In [2]:
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnableSequence

d:\project\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#Step 1: Load your dataset
loader=TextLoader("data/langchain_crewai_dataset.txt")
docs=loader.load()

splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks=splitter.split_documents(docs)

embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":4,"lambda_mult":0.7})

In [4]:
import os
from dotenv import load_dotenv
load_dotenv
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm=init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001B5B453D2B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B5B453DFD0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
#Step - 3 Query Decomposition
query_decomposition=PromptTemplate.from_template("""
You are an AI Assistant. Decompose the following complex question in 2 or 4 smaller simpler questions for better document retrieval.
                                                 
Question : "{question}"
                                            
Sub-questions:
""")

decomposition_chain=query_decomposition |llm |StrOutputParser()

In [6]:
query="What memory module does Langchain support and how are they different from CrewAI?"
decomposition_qa=decomposition_chain.invoke({"question":query})

In [7]:
print(decomposition_qa)

To break down the complex question into smaller, simpler questions for better document retrieval, I propose the following sub-questions:

1. **What type of memory modules does Langchain support?**
   This sub-question will help identify the specific memory modules supported by Langchain, such as memory-based models, graph-based models, or other types.

2. **What are the key characteristics of the memory modules supported by Langchain?**
   This sub-question will provide more insight into the features and functionalities of the memory modules supported by Langchain, such as their capacity, scalability, or integration capabilities.

3. **What are the memory modules supported by CrewAI?**
   This sub-question will help identify the memory modules supported by CrewAI, which can then be compared with those supported by Langchain.

4. **How do the memory modules supported by Langchain differ from those supported by CrewAI?**
   This sub-question will highlight the differences between the mem

In [9]:
#Step 4: qa chain for each subquestion
qa_prompt=PromptTemplate.from_template("""
Use the given context to answer the below questions.
                                
Context : {context}
                                       
Question:{input}
""")

qa_chain=create_stuff_documents_chain(llm=llm,prompt=qa_prompt)

In [11]:
# Full RAG Pipeline logic
def full_decomposition_rag_pipeline(user_query):
    sub_qs_text=decomposition_chain.invoke({"question":user_query})
    sub_qs=[q.strip("-•1234567890. ").strip() for q in sub_qs_text if q.strip()]

    results=[]
    for subq in sub_qs:
        docs=retriever.invoke(subq)
        result=qa_chain.invoke({"input":subq,"context":docs})
        results.append(f"Q:{subq}\n A:{result}")
    
    return "\n\n".join(results)

In [12]:
# Step 6 - Final Output
query="How does LangChain use memory and agents compared to CrewAI?"
res= full_decomposition_rag_pipeline(query)
print(f"Question:{query}")
print(F"Answer:{res}")

Question:How does LangChain use memory and agents compared to CrewAI?
Answer:Q:T
 A:It appears there's a lack of information in your question. Could you please rephrase or provide more context for question T so I can give an accurate response?

Q:o
 A:It seems like you forgot to ask any questions. Please provide a question related to the given context, and I'll do my best to help.

Q:b
 A:It seems like there are multiple questions. However, based on the given context, I can attempt to answer the following questions:

1. How does CrewAI enhance the accuracy and reduce hallucination of LLMs?
   Answer: By enabling knowledge to be fetched and injected into the LLM prompt to enhance accuracy and reduce hallucination.

2. What is the key feature of CrewAI in version 10?
   Answer: Dynamically communicating with one another.

3. What does CrewAI enable teams to build?
   Answer: Intelligent systems that scale both horizontally (more agents) and vertically (more reasoning depth).

4. What tec